In [0]:

import requests

# Citi Bike live station availability API
api_url = "https://gbfs.citibikenyc.com/gbfs/en/station_status.json"

# Send a request to the API
response = requests.get(api_url, timeout=20)

# Check whether the request was successful
response.raise_for_status()

print("API connection successful!")
print("Status code:", response.status_code)


API connection successful!
Status code: 200


In [0]:

# Convert the API response into Python data
data = response.json()

# Extract the station records
stations = data["data"]["stations"]

# Count the number of stations
print("Total stations received:", len(stations))

# Display the first station
print(stations[0])


Total stations received: 2519
{'station_id': '96e72113-3681-4d19-88bb-032e76720093', 'num_bikes_available': 0, 'num_bikes_disabled': 0, 'num_docks_available': 0, 'num_docks_disabled': 0, 'is_installed': 0, 'is_renting': 0, 'is_returning': 0, 'last_reported': 1795464185, 'legacy_id': '4736', 'num_ebikes_available': 0, 'eightd_has_available_keys': False}


In [0]:

# Convert the Python list into a PySpark DataFrame
df_bikes = spark.createDataFrame(stations)

# Count the records
print("Total records:", df_bikes.count())

# Display the first 5 rows
display(df_bikes.limit(5))

# Display the column names and data types
df_bikes.printSchema()


Total records: 2519


eightd_has_available_keys,is_installed,is_renting,is_returning,last_reported,legacy_id,num_bikes_available,num_bikes_disabled,num_docks_available,num_docks_disabled,num_ebikes_available,station_id,num_scooters_available,num_scooters_unavailable
false,0,0,0,1795464185,4736,0,0,0,0,0,96e72113-3681-4d19-88bb-032e76720093,null,null
false,0,0,0,1794416453,3404,0,0,0,0,0,66de5005-0aca-11e7-82f6-3863bb44ef7c,null,null
false,0,0,0,86400,465,0,0,0,0,0,66dc292c-0aca-11e7-82f6-3863bb44ef7c,null,null
false,0,0,0,86400,4915,0,0,0,0,0,475c44e1-31e3-4c60-b071-af2ca3f7618c,null,null
false,0,0,0,86400,4826,0,0,0,0,0,ba33c334-e17d-4478-ae3c-9c2f4dbfbb51,null,null


root
 |-- eightd_has_available_keys: boolean (nullable = true)
 |-- is_installed: long (nullable = true)
 |-- is_renting: long (nullable = true)
 |-- is_returning: long (nullable = true)
 |-- last_reported: long (nullable = true)
 |-- legacy_id: string (nullable = true)
 |-- num_bikes_available: long (nullable = true)
 |-- num_bikes_disabled: long (nullable = true)
 |-- num_docks_available: long (nullable = true)
 |-- num_docks_disabled: long (nullable = true)
 |-- num_ebikes_available: long (nullable = true)
 |-- station_id: string (nullable = true)
 |-- num_scooters_available: long (nullable = true)
 |-- num_scooters_unavailable: long (nullable = true)



In [0]:

import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

# Generate a unique ID for this API collection
snapshot_id = str(uuid.uuid4())

# Record the current time in UTC
snapshot_time = datetime.now(timezone.utc)

# Add tracking columns to the DataFrame
df_bronze = (
    df_bikes
    .withColumn("snapshot_id", F.lit(snapshot_id))
    .withColumn("ingestion_timestamp", F.lit(snapshot_time))
)

# Display the results
print("Snapshot ID:", snapshot_id)
print("Ingestion time:", snapshot_time)

display(
    df_bronze.select(
        "station_id",
        "num_bikes_available",
        "snapshot_id",
        "ingestion_timestamp"
    ).limit(5)
)


Snapshot ID: 4ef1552f-f762-46d0-b376-fdad1c74aabe
Ingestion time: 2026-11-24 19:10:53.049278+00:00


station_id,num_bikes_available,snapshot_id,ingestion_timestamp
96e72113-3681-4d19-88bb-032e76720093,0,4ef1552f-f762-46d0-b376-fdad1c74aabe,2026-11-24T19:10:53.049Z
66de5005-0aca-11e7-82f6-3863bb44ef7c,0,4ef1552f-f762-46d0-b376-fdad1c74aabe,2026-11-24T19:10:53.049Z
66dc292c-0aca-11e7-82f6-3863bb44ef7c,0,4ef1552f-f762-46d0-b376-fdad1c74aabe,2026-11-24T19:10:53.049Z
475c44e1-31e3-4c60-b071-af2ca3f7618c,0,4ef1552f-f762-46d0-b376-fdad1c74aabe,2026-11-24T19:10:53.049Z
ba33c334-e17d-4478-ae3c-9c2f4dbfbb51,0,4ef1552f-f762-46d0-b376-fdad1c74aabe,2026-11-24T19:10:53.049Z


In [0]:

from pyspark.sql import functions as F

table_name = "workspace.bronze.citibike_station_status"

# Count records before ingestion
rows_before = spark.table(table_name).count()

# Check whether this snapshot has already been saved
existing_snapshots = (
    spark.table(table_name)
    .filter(F.col("snapshot_id") == snapshot_id)
    .limit(1)
    .count()
)

if existing_snapshots == 0:
    # Add new records without deleting old ones
    df_bronze.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(table_name)

    print("New Bronze snapshot saved!")
else:
    print("This snapshot already exists. Skipping write.")

# Verify the table
rows_after = spark.table(table_name).count()

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("New rows added:", rows_after - rows_before)


New Bronze snapshot saved!
Rows before: 7557
Rows after: 10076
New rows added: 2519


In [0]:

from pyspark.sql import functions as F

df_history = spark.table(
    "workspace.bronze.citibike_station_status"
)

df_snapshot_history = (
    df_history
    .groupBy("snapshot_id", "ingestion_timestamp")
    .agg(
        F.count("*").alias("station_records")
    )
    .orderBy(F.col("ingestion_timestamp").desc())
)

display(df_snapshot_history)


snapshot_id,ingestion_timestamp,station_records
4ef1552f-f762-46d0-b376-fdad1c74aabe,2026-11-24T19:10:53.049Z,2519
fad61aad-facf-44cd-91b5-032a9052926a,2026-11-24T17:53:05.284Z,2519
791fccc8-a321-44c1-b799-005df9e12903,2026-11-20T21:24:03.186Z,2519
c494a760-1463-4766-8fe9-26e39b2c16c7,2026-11-20T21:23:38.351Z,2519
